### Load FIRMS data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

CLEANED_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed_data"
    / "firms_cleaned.csv"
)

df = pd.read_csv(CLEANED_FILE)

print("Shape:", df.shape)

Shape: (702968, 12)


### Recreate timestamp

In [2]:
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

time_str = (
    pd.to_numeric(
        df["acq_time"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

df["timestamp"] = pd.to_datetime(
    df["acq_date"].dt.strftime("%Y-%m-%d")
    + " "
    + time_str,
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

df = df.dropna(
    subset=[
        "latitude",
        "longitude",
        "timestamp",
        "frp"
    ]
).copy()

df = df.sort_values("timestamp").reset_index(drop=True)

print("Time range:")
print(df["timestamp"].min())
print(df["timestamp"].max())

Time range:
2025-01-01 03:50:00
2025-03-19 21:31:00


spatial cells

In [3]:

SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

df["lat_cell"] = (
    np.floor(
        df["latitude"] / LAT_CELL
    ).astype("int32")
)

df["lon_cell"] = (
    np.floor(
        df["longitude"] / LON_CELL
    ).astype("int32")
)

### Time-aware historical observation count

In [4]:
df["historical_detection_count"] = (
    df.groupby(
        ["lat_cell", "lon_cell"]
    ).cumcount()
)

print(
    df["historical_detection_count"].describe()
)

count    702968.000000
mean         22.686713
std          88.782966
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max        1571.000000
Name: historical_detection_count, dtype: float64


Previous FRP statistics

In [5]:
group = df.groupby(
    ["lat_cell", "lon_cell"]
)

df["historical_frp_sum"] = (
    group["frp"].cumsum() - df["frp"]
)

df["historical_frp_count"] = (
    df["historical_detection_count"]
)

df["historical_mean_frp"] = (
    df["historical_frp_sum"]
    / df["historical_frp_count"].replace(0, np.nan)
)

df["historical_mean_frp"] = (
    df["historical_mean_frp"]
    .fillna(0)
)

print(
    df["historical_mean_frp"].describe()
)

count    702968.000000
mean          6.390740
std          21.539452
min           0.000000
25%           0.000000
50%           1.936440
75%           5.235000
max        1909.200000
Name: historical_mean_frp, dtype: float64


### Current FRP deviation

In [6]:
df["local_frp_deviation"] = (
    df["frp"]
    - df["historical_mean_frp"]
)

print(
    df["local_frp_deviation"].describe()
)

count    702968.000000
mean          2.132876
std          24.466953
min       -1728.660000
25%          -0.656667
50%           1.170000
75%           4.160000
max        2056.400000
Name: local_frp_deviation, dtype: float64


### Previous-day activity


In [7]:
df["date_only"] = df["timestamp"].dt.date

In [8]:
## daily counts
daily_counts = (
    df.groupby(
        [
            "lat_cell",
            "lon_cell",
            "date_only"
        ]
    )
    .size()
    .reset_index(
        name="daily_count"
    )
)

daily_counts["date_only"] = pd.to_datetime(
    daily_counts["date_only"]
)

In [9]:
daily_counts = daily_counts.sort_values(
    [
        "lat_cell",
        "lon_cell",
        "date_only"
    ]
)

daily_counts["historical_daily_activity"] = (
    daily_counts
    .groupby(
        ["lat_cell", "lon_cell"]
    )["daily_count"]
    .cumsum()
    - daily_counts["daily_count"]
)

print(
    daily_counts.head()
)

   lat_cell  lon_cell  date_only  daily_count  historical_daily_activity
0       896      7753 2025-01-11            1                          0
1       896      7753 2025-01-12            1                          1
2       896      7753 2025-01-17            1                          2
3       896      7753 2025-03-01            2                          3
4       897      7749 2025-02-01            2                          0


### Merge historical daily activity

In [10]:
df["date_only"] = pd.to_datetime(
    df["date_only"]
)

df = df.merge(
    daily_counts[
        [
            "lat_cell",
            "lon_cell",
            "date_only",
            "historical_daily_activity"
        ]
    ],
    on=[
        "lat_cell",
        "lon_cell",
        "date_only"
    ],
    how="left"
)

df["historical_daily_activity"] = (
    df["historical_daily_activity"]
    .fillna(0)
)

print(
    df["historical_daily_activity"]
    .describe()
)

count    702968.000000
mean         21.491961
std          88.123884
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max        1551.000000
Name: historical_daily_activity, dtype: float64


### Recurrence indicator

In [11]:
df["previously_detected"] = (
    df["historical_detection_count"] > 0
).astype(int)

print(
    df["previously_detected"].value_counts()
)

previously_detected
1    445069
0    257899
Name: count, dtype: int64


#### Create the time-aware contextual dataset

In [12]:
context_columns = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

context_data = df[
    [
        "latitude",
        "longitude",
        "timestamp",
        "frp"
    ]
    + context_columns
].copy()

print(
    "Context dataset shape:",
    context_data.shape
)

context_data.head()

Context dataset shape: (702968, 9)


,latitude,longitude,timestamp,frp,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected
0,26.9361,94.0335,2025-01-01 03:50:00,6.5,0,0.0,6.5,0,0
1,25.3231,91.6008,2025-01-01 03:51:00,8.0,0,0.0,8.0,0,0
2,26.2760,92.8154,2025-01-01 03:51:00,11.6,0,0.0,11.6,0,0
3,25.3249,91.5900,2025-01-01 03:51:00,11.3,0,0.0,11.3,0,0
4,26.7587,92.7613,2025-01-01 03:51:00,5.3,0,0.0,5.3,0,0


#### Check for invalid values

In [13]:
print(
    context_data.isna().sum()
)

print("\nInfinite values:")

numeric_context = context_data[
    context_columns
]

print(
    np.isinf(
        numeric_context
        .select_dtypes(include=np.number)
    ).sum().sum()
)

latitude                      0
longitude                     0
timestamp                     0
frp                           0
historical_detection_count    0
historical_mean_frp           0
local_frp_deviation           0
historical_daily_activity     0
previously_detected           0
dtype: int64

Infinite values:
0


#### Save the leakage-safe version

In [14]:
FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "features"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TIME_AWARE_FILE = (
    FEATURE_DIR
    / "firms_context_features_time_aware.csv"
)

context_data.to_csv(
    TIME_AWARE_FILE,
    index=False
)

print("Saved successfully:")
print(TIME_AWARE_FILE)

Saved successfully:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\features\firms_context_features_time_aware.csv
